In [1]:
import pandas as pd 
import numpy as np
from sklearn.datasets import fetch_openml

In [2]:
df = fetch_openml('adult', version=2, as_frame=True).frame
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   age             48842 non-null  int64   
 1   workclass       46043 non-null  category
 2   fnlwgt          48842 non-null  int64   
 3   education       48842 non-null  category
 4   education-num   48842 non-null  int64   
 5   marital-status  48842 non-null  category
 6   occupation      46033 non-null  category
 7   relationship    48842 non-null  category
 8   race            48842 non-null  category
 9   sex             48842 non-null  category
 10  capital-gain    48842 non-null  int64   
 11  capital-loss    48842 non-null  int64   
 12  hours-per-week  48842 non-null  int64   
 13  native-country  47985 non-null  category
 14  class           48842 non-null  category
dtypes: category(9), int64(6)
memory usage: 2.7 MB


In [4]:
df = df.dropna()
df = df.drop_duplicates()

In [12]:
cat_cols = ['workclass', 'marital-status', 'occupation',
            'relationship', 'race', 'sex', 'native-country']
X = df.drop(columns=['class', 'fnlwgt', 'education'])
X = pd.get_dummies(X, columns=cat_cols)
X.head()

,age,education-num,capital-gain,capital-loss,hours-per-week,workclass_Federal-gov,workclass_Local-gov,workclass_Never-worked,workclass_Private,workclass_Self-emp-inc,...,native-country_Portugal,native-country_Puerto-Rico,native-country_Scotland,native-country_South,native-country_Taiwan,native-country_Thailand,native-country_Trinadad&Tobago,native-country_United-States,native-country_Vietnam,native-country_Yugoslavia
0,25,7,0,0,40,False,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False
1,38,9,0,0,50,False,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False
2,28,12,0,0,40,False,True,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,44,10,7688,0,40,False,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False
5,34,6,0,0,30,False,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False


In [13]:
y = (df['class'] == '>50K').astype(int)
y[:5]

0    0
1    0
2    1
3    1
5    0
Name: class, dtype: int64

In [14]:
X.shape

(45175, 88)

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [16]:
def gini(y):
    p = y.mean()
    return 1 - p**2 - (1 - p)**2

def entropy(y):
    p = np.array([1 - y.mean(), y.mean()])
    p = p[p > 0]                        
    return -np.sum(p * np.log2(p))

In [9]:
def best_split(features_n, X, y, criterion=gini):
    """features_n is the number of features that are going to be tested"""
    n = len(y)
    best_feature, best_t, best_score = None, None, float('inf')
    for i in range(features_n):
        x = X.iloc[:, i].astype(float)
        values = np.sort(x.unique())
        thresholds = (values[:-1] + values[1:]) / 2
        for t in thresholds:
            y_left  = y[x < t]
            y_right = y[x > t] 
            score = len(y_left)/n * criterion(y_left) + len(y_right)/n * criterion(y_right)
            if score < best_score:
                best_score = score
                best_t = t 
                best_feature = X.columns[i]
    return best_score, best_t, best_feature

In [17]:
def fit_stump(X, y, features_n):
    score, t, feature = best_split(features_n, X, y)
    y_left  = y[X[feature] < t]
    y_right = y[X[feature] > t]
    return {
        'feature':   feature,
        'threshold': t,
        'left':  int(y_left.mean()  > 0.5),  
        'right': int(y_right.mean() > 0.5),
    }

def predict(stump, X):
    go_left = X[stump['feature']] < stump['threshold']
    return np.where(go_left, stump['left'], stump['right'])

In [18]:
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

k = 5
stump = fit_stump(X_train, y_train, k)      
print(stump)

y_pred = predict(stump, X_test)             
print('My stump:', accuracy_score(y_test, y_pred))

sk = DecisionTreeClassifier(max_depth=1).fit(X_train.iloc[:, :k], y_train)
print('sklearn    :', accuracy_score(y_test, sk.predict(X_test.iloc[:, :k])))

print('Baseline   :', accuracy_score(y_test, np.zeros(len(y_test))))   

{'feature': 'capital-gain', 'threshold': np.float64(5119.0), 'left': 0, 'right': 1}
My stump: 0.7996679579413393
sklearn    : 0.7996679579413393
Baseline   : 0.7572772551189817
